# Depth 1 — Spectral mechanism: is the FC→SC asymmetry a low-rank, heritable backbone story?

**Setup**: The FC → SC > SC → FC asymmetry is robust at 10-seed precision across every
basis. *Why* it exists is unsettled. An earlier nugget: the cross-modal signal looked
concentrated in 1–3 SC components (SC-PC2 carried unique-to-FC R²≈0.26, then a cliff).
And separately: `pred_SC_raw` separates siblings at AUC=0.680 while `combined_pred_SC`
collapses to 0.505 (under diagnostic).

**The mechanistic question this notebook asks**: are these the *same* low-dim story?
Specifically, is there a small set of SC components that:
  (a) FC predicts well (high cross-modal R²),
  (b) separate relatives from strangers (heritable / family-structured), AND
  (c) carry the bulk of the asymmetry?

If yes, the narrative collapses into one sentence: **FC predicts a low-dimensional
heritable structural backbone, and that's why the asymmetry, the family signal, and
the modality dissociation all exist.**

If no — e.g. PC2 is spatially diffuse, or the heritable PCs are not the
FC-predictable PCs — that's a negative result on the mechanism. Still informative
("the asymmetry is broadly distributed, not concentrated"), but it tells a different
story.

## Analyses (parallel, single seed 0)

| Section | Question |
|---|---|
| A | Spatial mapping of SC-PC2: which edges / networks does it concentrate on? |
| B | Per-PC heritability: AUC(MZ/DZ/sibling vs unrelated_matched) for each of the top 10 SC PCs |
| C | Per-PC FC-predictability: R² of FC → each SC PC |
| D | **Synthesis cross-tab**: do the FC-predictable PCs coincide with the heritable PCs? |

Output: `../model_overviews/results/local_results/further_exploration/depth1_spectral_mechanism/`


In [2]:
# ============= SETUP =============
# Single source of truth for data + helpers is _setup.py next to this notebook.
# All notebooks in this directory share the same seed-0 split + closed-form helpers.
import sys
from pathlib import Path
# Jupyter launches at repo root, not notebook dir. Locate _setup.py by trying cwd
# then the known subpath; works on Torch (cwd=Conn2Conn) and laptop.
for _cand in [Path.cwd(),
              Path.cwd() / "notebooks-FC_to_SC-experimental" / "further_exploration",
              Path.cwd().parent,
              Path.cwd().parent.parent]:
    if (_cand / "_setup.py").exists():
        sys.path.insert(0, str(_cand))
        break
else:
    raise RuntimeError(f"_setup.py not found from cwd={Path.cwd()}")
from _setup import *  # load_seed_split, pca_pls_predict, combined_predict, br_per_component_predict,
                       # fit_basis_ols, full_panel_eval, pair_indices_by_relation,
                       # demeaned_cosine_pair_sim, extract_pair_sims, auc_vs_unrelated,
                       # results_dir; also np, pd, torch, PCA, BayesianRidge, LinearRegression

split = load_seed_split(seed=0)
base       = split["base"]
train_idx  = split["train_idx"]
test_idx   = split["test_idx"]
FC_tr, FC_te = split["FC_train"], split["FC_test"]
SC_tr, SC_te = split["SC_train"], split["SC_test"]
bv_tr, bv_te = split["bv_train"], split["bv_test"]
demo_tr, demo_te = split["demo_train"], split["demo_test"]
bvdemo_tr, bvdemo_te = split["bvdemo_train"], split["bvdemo_test"]
SC_train_mean = SC_tr.mean(axis=0)
FC_train_mean = FC_tr.mean(axis=0)

print(f"Setup OK. train={len(train_idx)}  test={len(test_idx)}  parc={PARCELLATION}")
print(f"  FC shape: {FC_tr.shape},  SC shape: {SC_tr.shape}")
print(f"  bv: {bv_tr.shape[1]}-dim, demo: {demo_tr.shape[1]}-dim, bv+demo: {bvdemo_tr.shape[1]}-dim")


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Setup OK. train=683  test=195  parc=Glasser
  FC shape: (683, 64620),  SC shape: (683, 64620)
  bv: 16-dim, demo: 10-dim, bv+demo: 26-dim


---

## Section A — Spatial mapping of SC-PC2

Fit PCA on `SC_train`, extract PC2 loadings (vector over the 64,620 upper-triangle edges),
identify top-K edges by `|loading|`. Edge IDs map back to (region_i, region_j) via the
upper-triangle indexing of the Glasser parcellation.

If PC2 concentrates on a coherent network (e.g. corpus callosum, default-mode commissural,
rich-club hubs), we have a candidate mechanism. If it's spatially random across the cortex,
the "low-dim backbone" story is shakier.


In [3]:
# ===== Section A: PCA on SC_train, extract PC2 loadings, top-K edges =====
K_PCA_DEPTH1 = 10  # top-10 PCs; PC2 is the headline but report a full table

pca_sc = PCA(n_components=K_PCA_DEPTH1, random_state=0).fit(SC_tr)
print(f"SC PCA fit. Explained var ratio (top {K_PCA_DEPTH1}):")
print("  PC#:  ratio   cumulative")
cum = 0.0
for i, r in enumerate(pca_sc.explained_variance_ratio_):
    cum += r
    print(f"  {i+1:3d}: {r:.4f}   {cum:.4f}")

# PC2 = component index 1 (0-indexed)
pc2_loadings = pca_sc.components_[1]  # shape (n_edges,)
print(f"\nPC2 loading vector: shape={pc2_loadings.shape}, "
      f"||·||={np.linalg.norm(pc2_loadings):.4f}, "
      f"max|·|={np.abs(pc2_loadings).max():.4f}")

# Top-K edges by |loading|.
TOP_K_EDGES = 50
order = np.argsort(-np.abs(pc2_loadings))
top_edge_idx = order[:TOP_K_EDGES]
print(f"\nTop {TOP_K_EDGES} edges by |PC2 loading|:")
print(f"  rank   edge_idx   loading")
for r, e in enumerate(top_edge_idx[:10]):
    print(f"  {r+1:4d}   {e:8d}   {pc2_loadings[e]:+.5f}")
print(f"  ... (full list saved to CSV)")

# Reconstruct (i, j) region pairs from the upper-triangle linear index.
# Assumes np.triu_indices(N, k=1) convention; N is inferred from edge count.
n_edges = pc2_loadings.shape[0]
N_regions = int((1 + np.sqrt(1 + 8 * n_edges)) / 2)
assert N_regions * (N_regions - 1) // 2 == n_edges, f"edge count {n_edges} not triangular"
print(f"\nInferred {N_regions} regions from {n_edges} edges.")
triu_i, triu_j = np.triu_indices(N_regions, k=1)

# Save edge-level table for the top edges.
out_dir = results_dir("depth1_spectral_mechanism")
top_df = pd.DataFrame({
    "rank":      np.arange(1, TOP_K_EDGES + 1),
    "edge_idx":  top_edge_idx,
    "region_i":  triu_i[top_edge_idx],
    "region_j":  triu_j[top_edge_idx],
    "loading":   pc2_loadings[top_edge_idx],
    "abs_load":  np.abs(pc2_loadings[top_edge_idx]),
})
top_df.to_csv(out_dir / "pc2_top_edges.csv", index=False)
print(f"\nSaved top edges -> {out_dir / 'pc2_top_edges.csv'}")

# Full loading vector for downstream.
np.save(out_dir / "sc_pc_loadings.npy", pca_sc.components_)
np.save(out_dir / "sc_pca_mean.npy", pca_sc.mean_)
print(f"Saved full PC loadings -> {out_dir / 'sc_pc_loadings.npy'} "
      f"(shape={pca_sc.components_.shape})")

# TODO: network-level aggregation. Needs Glasser parcellation labels mapped to network
# membership (e.g. Yeo7, Mesulam). Add when you decide which network atlas to use.
print("\nTODO: aggregate top edges by network membership (requires parcellation->network map).")


SC PCA fit. Explained var ratio (top 10):
  PC#:  ratio   cumulative
    1: 0.0383   0.0383
    2: 0.0162   0.0544
    3: 0.0149   0.0694
    4: 0.0136   0.0830
    5: 0.0118   0.0948
    6: 0.0111   0.1059
    7: 0.0105   0.1164
    8: 0.0094   0.1258
    9: 0.0088   0.1346
   10: 0.0087   0.1433

PC2 loading vector: shape=(64620,), ||·||=1.0000, max|·|=0.0706

Top 50 edges by |PC2 loading|:
  rank   edge_idx   loading
     1      13413   +0.07058
     2      31015   -0.06718
     3      13776   +0.06166
     4      30497   +0.06079
     5      51711   +0.05840
     6      12450   +0.05790
     7      57119   +0.05638
     8      12127   +0.05556
     9      63882   +0.05514
    10      51705   +0.05395
  ... (full list saved to CSV)

Inferred 360 regions from 64620 edges.

Saved top edges -> /scratch/ans9868/Conn2Conn/notebooks-FC_to_SC-experimental/model_overviews/results/local_results/further_exploration/depth1_spectral_mechanism/pc2_top_edges.csv
Saved full PC loadings -> /scratch

---

## Section B — Per-PC heritability

For each of the top 10 SC PCs, project `SC_test` onto that component (gives one score
per subject), build a pair similarity using `-|z_i - z_j|` (so closer = more similar),
and compute AUC(rel vs unrelated_matched) per relation bucket.

Same `pair_indices_by_relation` + `auc_vs_unrelated` as Analysis 1 in the main notebook;
just applied to single-component scores instead of demeaned edge vectors.

If PC2's heritability AUC is high (≥ 0.65 for siblings), it's a heritable component. If
it's near chance (0.5), the cross-modal signal lives in non-heritable structural variance.


In [4]:
# ===== Section B: per-PC heritability (AUC MZ/DZ/sib vs unrelated_matched) =====
SC_pc_scores_test = pca_sc.transform(SC_te)  # (n_test, K_PCA_DEPTH1)
print(f"SC test PC scores: shape={SC_pc_scores_test.shape}")

rng = np.random.default_rng(42)
pairs = pair_indices_by_relation(base.metadata_df, test_idx, rng, age_tol_yrs=3.0)
print(f"Pair counts: { {k: len(v) for k, v in pairs.items()} }")


def per_pc_aucs(score_vec, pairs_by_rel):
    """For a 1-D score vector, build sim = -|z_i - z_j| and AUC vs unrelated_matched."""
    # similarity: negative absolute difference (closer = higher sim, matches cosine convention sign)
    n = score_vec.shape[0]
    diff = np.abs(score_vec[:, None] - score_vec[None, :])  # (n, n)
    sim_matrix = -diff
    sims_by_rel = extract_pair_sims(sim_matrix, pairs_by_rel)
    return auc_vs_unrelated(sims_by_rel)


rows = []
for k in range(K_PCA_DEPTH1):
    aucs = per_pc_aucs(SC_pc_scores_test[:, k], pairs)
    rows.append({
        "pc":  k + 1,
        "explained_var_ratio": float(pca_sc.explained_variance_ratio_[k]),
        "AUC_MZ":      aucs.get("MZ", np.nan),
        "AUC_DZ":      aucs.get("DZ", np.nan),
        "AUC_sibling": aucs.get("sibling", np.nan),
    })

per_pc_heritability = pd.DataFrame(rows)
print("\nPer-PC heritability (AUC vs unrelated_matched):")
print(per_pc_heritability.to_string(index=False, float_format=lambda x: f"{x:7.4f}"))
per_pc_heritability.to_csv(out_dir / "per_pc_heritability.csv", index=False)
print(f"\nSaved -> {out_dir / 'per_pc_heritability.csv'}")


SC test PC scores: shape=(195, 10)
Pair counts: {'MZ': 33, 'DZ': 13, 'sibling': 125, 'unrelated_matched': 171}

Per-PC heritability (AUC vs unrelated_matched):
 pc  explained_var_ratio  AUC_MZ  AUC_DZ  AUC_sibling
  1               0.0383  0.8223  0.5038       0.4883
  2               0.0162  0.6413  0.4309       0.5781
  3               0.0149  0.7328  0.6019       0.5838
  4               0.0136  0.6929  0.7004       0.5619
  5               0.0118  0.6576  0.3473       0.5700
  6               0.0111  0.7622  0.6802       0.5559
  7               0.0105  0.6388  0.5254       0.5847
  8               0.0094  0.5712  0.5061       0.5707
  9               0.0088  0.6528  0.5416       0.4943
 10               0.0087  0.6716  0.5677       0.5831

Saved -> /scratch/ans9868/Conn2Conn/notebooks-FC_to_SC-experimental/model_overviews/results/local_results/further_exploration/depth1_spectral_mechanism/per_pc_heritability.csv


---

## Section C — Per-PC FC-predictability

For each top-10 SC PC, fit a regression FC → PC_k score using PCA(FC, 256) +
BayesianRidge (1-D target, single fit per PC, fast). Compute test R².

If FC-predictability concentrates in the same PCs that show heritability, the
mechanism is confirmed.


In [5]:
# ===== Section C: per-PC FC-predictability (R² of FC -> SC-PC_k) =====
from sklearn.metrics import r2_score

K_PCA_FC = 256
pca_fc = PCA(n_components=K_PCA_FC, random_state=0).fit(FC_tr)
Z_FC_tr = pca_fc.transform(FC_tr)
Z_FC_te = pca_fc.transform(FC_te)

# SC PC scores on train (target for fitting) and test (target for R²).
SC_pc_scores_train = pca_sc.transform(SC_tr)

rows_c = []
for k in range(K_PCA_DEPTH1):
    br = BayesianRidge(max_iter=300).fit(Z_FC_tr, SC_pc_scores_train[:, k])
    y_pred = br.predict(Z_FC_te)
    y_true = SC_pc_scores_test[:, k]
    r2 = r2_score(y_true, y_pred)
    # Also pearson for interpretability.
    pearson = float(np.corrcoef(y_pred, y_true)[0, 1])
    rows_c.append({"pc": k + 1, "FC_to_PC_R2": r2, "FC_to_PC_pearson": pearson})

per_pc_fcpred = pd.DataFrame(rows_c)
print("Per-PC FC-predictability:")
print(per_pc_fcpred.to_string(index=False, float_format=lambda x: f"{x:7.4f}"))
per_pc_fcpred.to_csv(out_dir / "per_pc_fc_predictability.csv", index=False)
print(f"\nSaved -> {out_dir / 'per_pc_fc_predictability.csv'}")


Per-PC FC-predictability:
 pc  FC_to_PC_R2  FC_to_PC_pearson
  1       0.5928            0.7738
  2       0.0004            0.1352
  3       0.2561            0.5306
  4       0.0905            0.3169
  5       0.0918            0.3037
  6       0.0338            0.2089
  7       0.0513            0.2882
  8       0.0685            0.2739
  9       0.0504            0.2309
 10       0.0778            0.2871

Saved -> /scratch/ans9868/Conn2Conn/notebooks-FC_to_SC-experimental/model_overviews/results/local_results/further_exploration/depth1_spectral_mechanism/per_pc_fc_predictability.csv


---

## Section D — Synthesis cross-tab

Merge B + C: for each PC, show explained variance + heritability AUC (sibling) +
FC-predictability R². Compute rank correlation between heritability AUC and
FC-predictability R² across PCs.

**Interpretation**:
- High positive correlation → the FC-predictable PCs ARE the heritable PCs. Mechanism story holds.
- No correlation → asymmetry and heritability are independent dimensions of SC variance.
- Negative correlation → FC predicts the *non*-heritable parts of SC. Surprising; worth digging.


In [6]:
# ===== Section D: synthesis cross-tab =====
synthesis = (
    per_pc_heritability[["pc", "explained_var_ratio", "AUC_MZ", "AUC_DZ", "AUC_sibling"]]
    .merge(per_pc_fcpred, on="pc")
)
print("Synthesis cross-tab:")
print(synthesis.to_string(index=False, float_format=lambda x: f"{x:7.4f}"))

# Rank correlation across PCs.
from scipy.stats import spearmanr
for h_col in ("AUC_MZ", "AUC_DZ", "AUC_sibling"):
    h = synthesis[h_col].values
    p = synthesis["FC_to_PC_R2"].values
    mask = ~(np.isnan(h) | np.isnan(p))
    if mask.sum() >= 3:
        rho, pval = spearmanr(h[mask], p[mask])
        print(f"  Spearman({h_col}, FC_to_PC_R2)  rho={rho:+.3f}  p={pval:.3f}")
    else:
        print(f"  Spearman({h_col}, FC_to_PC_R2)  -- insufficient non-nan PCs")

synthesis.to_csv(out_dir / "synthesis_cross_tab.csv", index=False)
print(f"\nSaved -> {out_dir / 'synthesis_cross_tab.csv'}")

# PC2 spotlight (the original nugget).
pc2 = synthesis[synthesis["pc"] == 2].iloc[0]
print(f"\nPC2 spotlight: var={pc2['explained_var_ratio']:.4f}  "
      f"AUC_sib={pc2['AUC_sibling']:.4f}  FC->PC2 R²={pc2['FC_to_PC_R2']:.4f}")
print("\nVerdict template (fill in once you see the numbers):")
print("  - If PC2 AUC_sib >= 0.65 AND FC->PC2 R² >= 0.15 AND spearman > 0.5:")
print("       -> mechanism CONFIRMED: low-dim heritable backbone story.")
print("  - If PC2 AUC_sib ~ 0.5 OR spearman ~ 0:")
print("       -> mechanism REJECTED: asymmetry and heritability are independent.")
print("  - If FC->PC2 R² is high but AUC_sib low:")
print("       -> mechanism PARTIAL: FC predicts a non-heritable dominant mode.")


Synthesis cross-tab:
 pc  explained_var_ratio  AUC_MZ  AUC_DZ  AUC_sibling  FC_to_PC_R2  FC_to_PC_pearson
  1               0.0383  0.8223  0.5038       0.4883       0.5928            0.7738
  2               0.0162  0.6413  0.4309       0.5781       0.0004            0.1352
  3               0.0149  0.7328  0.6019       0.5838       0.2561            0.5306
  4               0.0136  0.6929  0.7004       0.5619       0.0905            0.3169
  5               0.0118  0.6576  0.3473       0.5700       0.0918            0.3037
  6               0.0111  0.7622  0.6802       0.5559       0.0338            0.2089
  7               0.0105  0.6388  0.5254       0.5847       0.0513            0.2882
  8               0.0094  0.5712  0.5061       0.5707       0.0685            0.2739
  9               0.0088  0.6528  0.5416       0.4943       0.0504            0.2309
 10               0.0087  0.6716  0.5677       0.5831       0.0778            0.2871
  Spearman(AUC_MZ, FC_to_PC_R2)  rho=+0.491 